In [ ]:
# installing needed dependencies on colab
!pip install unsloth torch trl evaluate transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Load necessary libraries for GRPO
import unsloth

from trl import GRPOConfig, GRPOTrainer
from transformers import TrainingArguments,GenerationConfig
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel

import torch
from datasets import load_dataset

from evaluate import load
from tqdm import tqdm
import json


print("GRPO libraries loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GRPO libraries loaded successfully!


In [ ]:
"""
GRPO Training for LLM Agent in Iterated Prisoner's Dilemma (IPD)
against fixed rule-based opponents (TFT, Always Defect, Always Cooperate, Random).

Requirements:
    pip install trl transformers torch datasets peft

Tested with: trl>=0.12, transformers>=4.40
"""

import random
import torch
from dataclasses import dataclass, field
from typing import Literal
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import GRPOConfig, GRPOTrainer


# ─────────────────────────────────────────────
# 1.  OPPONENT STRATEGIES
# ─────────────────────────────────────────────

OpponentType = Literal["tft", "always_defect", "always_cooperate", "random"]

class Opponent:
    """Stateless, rule-based opponent.  my_last / opp_last are the moves
    from the *previous* step (None at the very first step)."""

    def __init__(self, strategy: OpponentType, cooperate_prob: float = 0.5):
        self.strategy = strategy
        self.cooperate_prob = cooperate_prob  # used only by 'random'

    def act(self, my_last: str | None, opp_last: str | None) -> str:
        """
        my_last  – opponent's own last move  (C or D)
        opp_last – LLM agent's last move    (C or D)
        """
        if self.strategy == "always_cooperate":
            return "C"
        if self.strategy == "always_defect":
            return "D"
        if self.strategy == "random":
            return "C" if random.random() < self.cooperate_prob else "D"
        if self.strategy == "tft":
            # First move: cooperate; then mirror opponent's last move
            return "C" if opp_last is None else opp_last
        raise ValueError(f"Unknown strategy: {self.strategy}")


# ─────────────────────────────────────────────
# 2.  IPD PAYOFF MATRIX
# ─────────────────────────────────────────────

# Payoff for (agent_action, opponent_action) -> agent_reward
PAYOFF = {
    ("C", "C"): 3,
    ("C", "D"): 0,
    ("D", "C"): 4,
    ("D", "D"): 1,
}

def get_reward(agent_action: str, opp_action: str) -> float:
    return float(PAYOFF.get((agent_action, opp_action), -6.0))  # -6 for illegal


# ─────────────────────────────────────────────
# 3.  PROMPT BUILDER
# ─────────────────────────────────────────────

ACTION_TOKENS = {"C": "action1", "D": "action2"}  # hide the real meaning

def build_prompt(agent_last: str | None, opp_last: str | None) -> str:
    """
    Build an IPD prompt that encodes the last moves without revealing
    'Prisoner's Dilemma' semantics (following the paper's approach).
    """
    history_str = (
        "This is your first move."
        if agent_last is None
        else (
            f"Last time, you played {ACTION_TOKENS[agent_last]} "
            f"and your opponent played {ACTION_TOKENS[opp_last]}, "
            f"so you got {PAYOFF[(agent_last, opp_last)]} points."
        )
    )

    prompt = (
        "You are playing a game against another agent. "
        "You must choose either action1 or action2. "
        "Points are awarded as follows (you are the row player):\n\n"
        "          | action1 | action2 |\n"
        "| action1 |   3, 3  |   0, 4  |\n"
        "| action2 |   4, 0  |   1, 1  |\n\n"
        f"{history_str}\n\n"
        "What action do you take to achieve the highest possible score? "
        "Reply with ONLY 'action1' or 'action2'. Do not explain."
    )
    return prompt


# ─────────────────────────────────────────────
# 4.  DATASET GENERATOR  (dynamic, online)
# ─────────────────────────────────────────────

def generate_episode_dataset(
    opponent: Opponent,
    n_prompts: int = 256,
) -> Dataset:
    """
    Generate a batch of independent game states to use as prompts.
    Each row is a single-step snapshot: we sample a random prior state
    (or first move) and record the ground-truth opponent action for the
    reward function to use.

    The reward function receives the generated token (action1 / action2),
    looks up the opponent's response, and returns the payoff.

    Note: for simplicity, each prompt is an independent 1-step interaction.
    For true n-step episodes use a stateful environment wrapper instead.
    """
    rows = []
    prior_states = [
        (None, None),      # first move
        ("C", "C"),
        ("C", "D"),
        ("D", "C"),
        ("D", "D"),
    ]

    for _ in range(n_prompts):
        agent_last, opp_last = random.choice(prior_states)
        opp_action = opponent.act(my_last=opp_last, opp_last=agent_last)
        prompt = build_prompt(agent_last, opp_last)

        rows.append({
            "prompt": prompt,
            "opp_action": opp_action,   # stored for reward computation
        })

    return Dataset.from_list(rows)


# ─────────────────────────────────────────────
# 5.  REWARD FUNCTION
# ─────────────────────────────────────────────

ILLEGAL_REWARD = -6.0

def make_reward_fn(dataset: Dataset):
    """
    Returns a reward function compatible with GRPOTrainer.
    GRPOTrainer calls:
        reward_fn(prompts, completions, **kwargs) -> list[float]
    """
    # Build a lookup from prompt text -> opp_action
    prompt_to_opp = {row["prompt"]: row["opp_action"] for row in dataset}

    def reward_fn(prompts: list[str], completions: list[str], **kwargs) -> list[float]:
        rewards = []
        for prompt, completion in zip(prompts, completions):
            opp_action = prompt_to_opp.get(prompt, "C")  # fallback
            text = completion.strip().lower()

            if "action1" in text:
                agent_action = "C"
            elif "action2" in text:
                agent_action = "D"
            else:
                rewards.append(ILLEGAL_REWARD)
                continue

            rewards.append(get_reward(agent_action, opp_action))

        return rewards

    return reward_fn


# ─────────────────────────────────────────────
# 6.  MORAL REWARD VARIANTS  (optional)
# ─────────────────────────────────────────────

def make_moral_reward_fn(dataset: Dataset, reward_type: str = "game", omega: float = 3.0):
    """
    reward_type: 'game' | 'deontological' | 'utilitarian' | 'game+deontological'
    """
    prompt_to_opp = {row["prompt"]: row["opp_action"] for row in dataset}

    def reward_fn(prompts, completions, **kwargs):
        rewards = []
        for prompt, completion in zip(prompts, completions):
            opp_action = prompt_to_opp.get(prompt, "C")
            text = completion.strip().lower()

            if "action1" in text:
                agent_action = "C"
            elif "action2" in text:
                agent_action = "D"
            else:
                rewards.append(ILLEGAL_REWARD)
                continue

            r_agent = get_reward(agent_action, opp_action)
            r_opp   = get_reward(opp_action, agent_action)

            if reward_type == "game":
                r = r_agent

            elif reward_type == "deontological":
                # Penalize defecting against a cooperator
                r = -omega if (agent_action == "D" and opp_action == "C") else 0.0

            elif reward_type == "utilitarian":
                r = r_agent + r_opp  # social welfare

            elif reward_type == "game+deontological":
                penalty = -omega if (agent_action == "D" and opp_action == "C") else 0.0
                r = r_agent + penalty

            else:
                raise ValueError(f"Unknown reward_type: {reward_type}")

            rewards.append(r)
        return rewards

    return reward_fn


# ─────────────────────────────────────────────
# 7.  MAIN TRAINING LOOP
# ─────────────────────────────────────────────

@dataclass
class TrainingConfig:
    model_name: str = "google/gemma-2-2b-it"
    opponent_strategy: OpponentType = "tft"
    reward_type: str = "game"           # game | deontological | utilitarian | game+deontological
    n_episodes: int = 100               # outer loop iterations
    prompts_per_episode: int = 64       # prompts generated each episode
    grpo_group_size: int = 8            # G in GRPO (samples per prompt)
    max_new_tokens: int = 8             # completions are very short
    learning_rate: float = 1e-5
    kl_coef: float = 0.1
    lora_rank: int = 64
    output_dir: str = "./grpo_ipd_output"
    use_4bit: bool = True


def train(cfg: TrainingConfig):

    # ── Model & tokenizer ──────────────────────────────────────────────
    print(f"Loading model: {cfg.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {}
    if cfg.use_4bit:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    base_model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        **model_kwargs,
    )

    # ── LoRA ───────────────────────────────────────────────────────────
    lora_config = LoraConfig(
        r=cfg.lora_rank,
        lora_alpha=cfg.lora_rank * 2,
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    # ── Opponent ───────────────────────────────────────────────────────
    opponent = Opponent(strategy=cfg.opponent_strategy)

    # ── GRPO config ────────────────────────────────────────────────────
    grpo_cfg = GRPOConfig(
        output_dir=cfg.output_dir,
        num_train_epochs=1,              # we control episodes in the outer loop
        per_device_train_batch_size=cfg.grpo_group_size,
        gradient_accumulation_steps=4,
        learning_rate=cfg.learning_rate,
        kl_coef=cfg.kl_coef,
        max_new_tokens=cfg.max_new_tokens,
        num_generations=cfg.grpo_group_size,  # G
        temperature=1.0,
        logging_steps=1,
        save_strategy="no",
        report_to="none",
    )

    # ── Training loop  ─────────────────────────────────────────────────
    print(f"\nStarting GRPO training | opponent={cfg.opponent_strategy} | reward={cfg.reward_type}")
    print(f"Episodes: {cfg.n_episodes}, prompts/episode: {cfg.prompts_per_episode}\n")

    for episode in range(cfg.n_episodes):

        # Generate fresh prompts for this episode
        dataset = generate_episode_dataset(opponent, n_prompts=cfg.prompts_per_episode)

        reward_fn = make_moral_reward_fn(
            dataset,
            reward_type=cfg.reward_type,
            omega=3.0,
        )

        # One GRPO update on this episode's dataset
        trainer = GRPOTrainer(
            model=model,
            args=grpo_cfg,
            processing_class=tokenizer,
            train_dataset=dataset.select_columns(["prompt"]),
            reward_funcs=reward_fn,
        )

        trainer.train()

        # ── Quick eval ─────────────────────────────────────────────────
        if (episode + 1) % 10 == 0:
            coop_rate = evaluate_agent(model, tokenizer, opponent, n_games=50)
            print(f"Episode {episode + 1:>4}/{cfg.n_episodes} | "
                  f"Cooperation rate: {coop_rate:.2%}")

    print("\nTraining complete. Saving adapter...")
    model.save_pretrained(cfg.output_dir)
    tokenizer.save_pretrained(cfg.output_dir)
    print(f"Saved to {cfg.output_dir}")
    return model, tokenizer


# ─────────────────────────────────────────────
# 8.  EVALUATION
# ─────────────────────────────────────────────

@torch.no_grad()
def evaluate_agent(
    model,
    tokenizer,
    opponent: Opponent,
    n_games: int = 100,
    n_steps: int = 10,
) -> float:
    """
    Run n_games independent games of n_steps each.
    Returns the average cooperation rate of the LLM agent.
    """
    model.eval()
    cooperations = 0
    total_steps = 0

    for _ in range(n_games):
        agent_last = opp_last = None

        for _ in range(n_steps):
            prompt = build_prompt(agent_last, opp_last)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            out = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                temperature=None,
                top_p=None,
            )
            completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            text = completion.strip().lower()

            agent_action = "C" if "action1" in text else "D"
            opp_action = opponent.act(my_last=opp_last, opp_last=agent_last)

            if agent_action == "C":
                cooperations += 1
            total_steps += 1

            agent_last = agent_action
            opp_last = opp_action

    model.train()
    return cooperations / total_steps if total_steps > 0 else 0.0


# ─────────────────────────────────────────────
# 9.  ENTRYPOINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="GRPO IPD Training")
    parser.add_argument("--model",    default="google/gemma-2-2b-it")
    parser.add_argument("--opponent", default="tft",
                        choices=["tft", "always_defect", "always_cooperate", "random"])
    parser.add_argument("--reward",   default="game",
                        choices=["game", "deontological", "utilitarian", "game+deontological"])
    parser.add_argument("--episodes", type=int, default=100)
    parser.add_argument("--no-4bit",  action="store_true")
    parser.add_argument("--output",   default="./grpo_ipd_output")
    args = parser.parse_args()

    cfg = TrainingConfig(
        model_name=args.model,
        opponent_strategy=args.opponent,
        reward_type=args.reward,
        n_episodes=args.episodes,
        use_4bit=not args.no_4bit,
        output_dir=args.output,
    )

    train(cfg)